In [1]:
import pandas as pd
from src.config import BINARY_RESULTS_DIR as R, RESULTS_DIR

def best(df, metric):
    """Best model of every experiment."""
    return df.loc[df.groupby("experiment", sort=False)[metric].idxmax()]

COLS = ["experiment", "model", "features", "train_images", "accuracy"]

In [2]:
# Table 1 - binary, CV on the subset
FEATURES_1 = {"E0 all features": "base (230)", "E1 colour only": "colour only (132)", "E1 texture only": "texture only (98)",
              "E2 top-50": "top-50 of base", "E2 top-100": "top-100 of base", "E3 PCA 95%": "PCA 95% of base",
              "E4 no hair removal": "base (230), no hair removal", "E5 ensembles": "base (230)"}
t1 = pd.concat([best(pd.read_csv(R / "exp0123_feature_selection.csv"), "accuracy"),
                pd.read_csv(R / "exp4_no_hair.csv"),
                best(pd.read_csv(R / "exp5_binary_ensembles.csv").query("experiment == 'E5 ensembles'"), "accuracy")])
t1 = t1.assign(features=t1["experiment"].map(FEATURES_1), train_images=4136)
t1[COLS].round(3)

,experiment,model,features,train_images,accuracy
3,E0 all features,SVM (RBF),base (230),4136,0.858
11,E1 colour only,CatBoost,colour only (132),4136,0.841
15,E1 texture only,SVM (RBF),texture only (98),4136,0.821
21,E2 top-50,SVM (RBF),top-50 of base,4136,0.825
27,E2 top-100,SVM (RBF),top-100 of base,4136,0.854
33,E3 PCA 95%,SVM (RBF),PCA 95% of base,4136,0.853
0,E4 no hair removal,SVM (RBF),"base (230), no hair removal",4136,0.858
5,E5 ensembles,Stacking (LogReg),base (230),4136,0.859


In [3]:
# Table 2 - binary, CV on the full train set
e6 = (pd.read_csv(R / "exp6_full_train_folds.csv")
      .groupby(["model", "train data"], sort=False)[["n_train", "accuracy"]].mean().reset_index())
e6 = e6.assign(experiment="E6 more data", features="base (230)", train_images=e6["n_train"].round().astype(int))

FEATURES_2 = {"F0 base": "base (230)", "F1 base + regional": "base + regional (442)", "F2 base + HoG": "base + HoG (554)",
              "F3 all": "base + regional + HoG (766)", "F4 all + SelectKBest": "SelectKBest of 766"}
e7 = pd.read_csv(R / "exp7_new_features.csv")
e7 = e7.assign(features=e7["experiment"].map(FEATURES_2), train_images=12156)

e8 = pd.DataFrame([{"experiment": "F5 ensemble", "model": "Stacking (LogReg)", "features": "base + regional (442)",
                    "train_images": 12156, "accuracy": pd.read_csv(R / "exp8_f1_stacking_folds.csv")["accuracy"].mean()}])
t2 = pd.concat([e6, e7, e8], ignore_index=True)
t2[COLS].round(3)

,experiment,model,features,train_images,accuracy
0,E6 more data,Stacking (LogReg),base (230),12156,0.837
1,E6 more data,Stacking (LogReg),base (230),4136,0.819
2,E6 more data,SVM (RBF),base (230),12156,0.830
3,E6 more data,SVM (RBF),base (230),4136,0.810
4,F0 base,SVM (RBF),base (230),12156,0.831
5,F1 base + regional,SVM (RBF),base + regional (442),12156,0.848
6,F2 base + HoG,SVM (RBF),base + HoG (554),12156,0.827
7,F3 all,SVM (RBF),base + regional + HoG (766),12156,0.842
8,F4 all + SelectKBest,SVM (RBF),SelectKBest of 766,12156,0.840
9,F5 ensemble,Stacking (LogReg),base + regional (442),12156,0.852


In [4]:
# Table 3 - final binary model on validation
pd.read_csv(RESULTS_DIR / "final_binary_validation.csv")[["model", "features", "train_images", "accuracy"]].round(3)

,model,features,train_images,accuracy
0,Stacking (F1),base + regional (442),15195,0.863
